# M1 분류 전처리

이벤트당 1행, 7일 윈도우, M1 공통 10개 센서 기준으로 feature CSV를 만든다.

In [1]:
from pathlib import Path
import math
import zipfile

import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / '05_데이터셋' / 'PreDist'
ZIP_PATH = DATA_DIR / 'predist_dataset.zip'
OUTPUT_DIR = ROOT / '07_데이터산출물'
EVENT_INDEX_PATH = OUTPUT_DIR / 'm1_classification_event_index.csv'
FEATURES_PATH = OUTPUT_DIR / 'm1_classification_features.csv'

COMMON_SENSOR_COLUMNS = [
    'outdoor_temperature',
    's_hc1_supply_temperature',
    's_hc1_supply_temperature_setpoint',
    'p_hc1_return_temperature',
    'p_net_meter_energy',
    'p_net_meter_volume',
    'p_net_meter_heat_power',
    'p_net_meter_flow',
    'p_net_supply_temperature',
    'p_net_return_temperature',
]

event_index = pd.read_csv(EVENT_INDEX_PATH)
event_index.head()

,sample_id,manufacturer,label,source_file,source_event_id,substation_id,window_start,window_end,window_days,window_policy,report_date,possible_anomaly_start,possible_anomaly_end,training_start,training_end
0,efd_possible_0001,manufacturer_1,efd_possible,faults.csv,1,10,2014-04-27 14:44:00,2014-05-04 14:44:00,7.0,report_date_minus_7d_to_report_date,2014-05-04 14:44:00,2014-05-03 16:00:00,2014-05-05 04:00:00,2012-03-28 09:00:00,2014-04-20 14:44:00
1,efd_possible_0003,manufacturer_1,efd_possible,faults.csv,3,12,2015-11-24 10:56:00,2015-12-01 10:56:00,7.0,report_date_minus_7d_to_report_date,2015-12-01 10:56:00,2015-11-29 12:00:00,2015-12-02 10:56:00,2015-03-01 00:00:00,2015-11-17 10:56:00
2,efd_possible_0040,manufacturer_1,efd_possible,faults.csv,40,24,2016-03-31 07:47:00,2016-04-07 07:47:00,7.0,report_date_minus_7d_to_report_date,2016-04-07 07:47:00,2016-04-05 13:00:00,2016-04-08 07:47:00,2015-11-20 00:00:00,2016-03-24 07:47:00
3,efd_possible_0052,manufacturer_1,efd_possible,faults.csv,52,21,2016-12-05 15:55:00,2016-12-12 15:55:00,7.0,report_date_minus_7d_to_report_date,2016-12-12 15:55:00,2016-12-11 20:00:00,2016-12-13 15:55:00,2015-11-30 09:00:00,2016-11-28 15:55:00
4,efd_possible_0067,manufacturer_1,efd_possible,faults.csv,67,7,2017-11-19 13:15:00,2017-11-26 13:15:00,7.0,report_date_minus_7d_to_report_date,2017-11-26 13:15:00,2017-04-23 01:00:00,2017-11-27 13:15:00,2015-04-12 00:00:00,2017-04-11 00:00:00


In [2]:
def load_substation_frame(zf: zipfile.ZipFile, substation_id: int) -> pd.DataFrame:
    path = f'manufacturer 1/operational_data/substation_{substation_id}.csv'
    with zf.open(path) as f:
        df = pd.read_csv(f, sep=';', usecols=['timestamp', *COMMON_SENSOR_COLUMNS])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    for col in COMMON_SENSOR_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def add_sensor_stats(feature_row: dict, window: pd.DataFrame, sample_count: int) -> dict:
    for col in COMMON_SENSOR_COLUMNS:
        series = window[col] if sample_count else pd.Series(dtype='float64')
        valid = series.dropna()
        feature_row[f'{col}__mean'] = float(series.mean()) if sample_count else math.nan
        feature_row[f'{col}__std'] = float(series.std()) if sample_count else math.nan
        feature_row[f'{col}__min'] = float(series.min()) if sample_count else math.nan
        feature_row[f'{col}__max'] = float(series.max()) if sample_count else math.nan
        feature_row[f'{col}__median'] = float(series.median()) if sample_count else math.nan
        feature_row[f'{col}__last_minus_first'] = float(valid.iloc[-1] - valid.iloc[0]) if len(valid) >= 2 else math.nan
        feature_row[f'{col}__missing_rate'] = float(series.isna().mean()) if sample_count else math.nan
    return feature_row

In [3]:
feature_rows = []
cache = {}
expected_count = 7 * 24 * 6

with zipfile.ZipFile(ZIP_PATH) as zf:
    for _, row in event_index.iterrows():
        substation_id = int(row['substation_id'])
        if substation_id not in cache:
            cache[substation_id] = load_substation_frame(zf, substation_id)
        df = cache[substation_id]
        start = pd.to_datetime(row['window_start'])
        end = pd.to_datetime(row['window_end'])
        window = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]

        sample_count = int(len(window))
        feature_row = row.to_dict()
        feature_row['sample_count'] = sample_count
        feature_row['expected_sample_count'] = expected_count
        feature_row['coverage_rate'] = round(sample_count / expected_count, 6)
        feature_rows.append(add_sensor_stats(feature_row, window, sample_count))

features = pd.DataFrame(feature_rows)
features.to_csv(FEATURES_PATH, index=False, encoding='utf-8-sig')
features[['sample_id', 'label', 'substation_id', 'sample_count', 'coverage_rate']].head()

,sample_id,label,substation_id,sample_count,coverage_rate
0,efd_possible_0001,efd_possible,10,1008,1.00000
1,efd_possible_0003,efd_possible,12,1008,1.00000
2,efd_possible_0040,efd_possible,24,1003,0.99504
3,efd_possible_0052,efd_possible,21,1008,1.00000
4,efd_possible_0067,efd_possible,7,1008,1.00000


In [4]:
assert len(features) == 50
assert features['label'].value_counts().to_dict() == {'normal': 35, 'efd_possible': 15}
assert (features['sample_count'] > 0).all()
assert set(features['manufacturer']) == {'manufacturer_1'}
assert set(features['window_days'].round(6)) == {7.0}
expected_feature_columns = {f'{col}__{stat}' for col in COMMON_SENSOR_COLUMNS for stat in ['mean', 'std', 'min', 'max', 'median', 'last_minus_first', 'missing_rate']}
assert expected_feature_columns.issubset(set(features.columns))
features.groupby('label')['coverage_rate'].agg(['min', 'median', 'max']).round(4)

,min,median,max
label,,,
efd_possible,0.7242,1.0,1.0
normal,1.0000,1.0,1.0
